# RL with Less
This project attempts to implement the methods described in the paper ["Learning more with Less"](https://arxiv.org/abs/2501.10499). Primary task is to train a robot to trace an elliptical path with its end-effector, a non-trivial task. Towards the end, I will explore how to automate reward functions following strategies outlined in [ARCHIE: Autonomous Reinforcement Learning with GPT-4](https://arxiv.org/abs/2503.04280)

In [ ]:
import gymnasium as gym
import numpy as np
import mujoco
from mujoco import mjr_render
import mujoco.viewer

In [ ]:
# preferred initial state
"""
<key
  time="16.078"
  qpos="-0.0821719 -3.15863e-05 0.587366 0.999675 0.000538771 -0.0254943 6.27497e-05 0.00798042 -0.0345756 -0.294294 -0.00797118 -0.0346344 -0.295851 0.0213032 -0.00719754 -0.304018 -0.0214776 -0.00772524 -0.306353 -2.13996e-05 -3.13773 3.1343 -0.00178122 0.00274217 -5.84961e-07 0.000185415"
  qvel="0.000357176 -6.32214e-06 -0.000183815 1.34908e-05 0.000420337 -1.02752e-05 0.00086477 -0.000869234 -0.000465488 -0.000850424 -0.000906848 -0.000466469 0.000707315 0.000969623 0.000628213 -0.00072065 0.000996345 0.000606928 1.87757e-07 -8.84173e-05 -1.87744e-05 0.000429102 -2.37872e-08 -9.9795e-08 -1.57788e-08"
  ctrl="0 0 0 0 0 0 0 0 0 0 0 0 0 -3.142 3.142 0 0 0 0"
/>
"""




In [ ]:

model = mujoco.MjModel.from_xml_path("../robots/boston_dynamics_spot/scene_arm.xml")
data = mujoco.MjData(model)

crt_init = crl = np.zeros(model.nu)
crt_init[13] = -3.142
crt_init[14] = 3.142

data.ctrl[:] = crt_init
mujoco.mj_forward(model, data)

mujoco.viewer.launch(model,data)

: 

In [ ]:
data.actuator("arm_sh1")

<_MjDataActuatorViews
  ctrl: array([0.])
  force: array([-82.71807168])
  id: 13
  length: array([0.16708232])
  moment: array([ 0.02641499,  0.06222613,  0.20147409,  0.03975649,  0.22512304,
       -0.00620759, -0.36013765,  0.01540244,  0.02392648, -0.00282821,
        0.00995489,  0.01616649, -0.0042311 , -0.02831094,  0.00064121,
       -0.05631449,  0.12842801, -0.14851834, -0.12343987,  0.14615802,
       -0.32161648,  0.34105272,  0.01579818,  0.13010581, -0.02637893])
  name: 'arm_sh1'
  velocity: array([-0.02057722])
>

In [65]:
model.actuator("arm_sh1")

<_MjModelActuatorViews
  acc0: array([29.18345455])
  actadr: array([-1], dtype=int32)
  actlimited: array([0], dtype=uint8)
  actnum: array([0], dtype=int32)
  actrange: array([0., 0.])
  biasprm: array([   0., -500.,  -40.,    0.,    0.,    0.,    0.,    0.,    0.,
          0.])
  biastype: array([1], dtype=int32)
  cranklength: array([0.])
  ctrllimited: array([1], dtype=uint8)
  ctrlrange: array([-3.14159 ,  0.523599])
  dynprm: array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
  dyntype: array([0], dtype=int32)
  forcelimited: array([0], dtype=uint8)
  forcerange: array([0., 0.])
  gainprm: array([500.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.])
  gaintype: array([0], dtype=int32)
  gear: array([1., 0., 0., 0., 0., 0.])
  group: array([0], dtype=int32)
  id: 13
  length0: array([0.])
  lengthrange: array([0., 0.])
  name: 'arm_sh1'
  trnid: array([14, -1], dtype=int32)
  trntype: array([0], dtype=int32)
  user: array([], dtype=float64)
>

In [59]:
model.joint("arm_sh1")

<_MjModelJointViews
  M0: array([1.71830004])
  Madr: array([124], dtype=int32)
  armature: array([0.])
  axis: array([0., 1., 0.])
  bodyid: array([15], dtype=int32)
  damping: array([0.])
  dofadr: array([19], dtype=int32)
  frictionloss: array([0.])
  group: array([0], dtype=int32)
  id: 14
  invweight0: array([13.05646922])
  jntid: array([14], dtype=int32)
  limited: array([1], dtype=uint8)
  margin: array([0.])
  name: 'arm_sh1'
  parentid: array([18], dtype=int32)
  pos: array([0., 0., 0.])
  qpos0: array([0.])
  qpos_spring: array([0.])
  qposadr: array([20], dtype=int32)
  range: array([-3.14159 ,  0.523599])
  simplenum: array([0], dtype=int32)
  solimp: array([9.0e-01, 9.5e-01, 1.0e-03, 5.0e-01, 2.0e+00])
  solref: array([0.02, 1.  ])
  stiffness: array([0.])
  type: array([3], dtype=int32)
  user: array([], dtype=float64)
>

In [55]:
from mujoco import mjtJoint
mjtJoint.mjJNT_HINGE

<mjtJoint.mjJNT_HINGE: 3>

In [34]:
data.qpos = qpos

In [ ]:
len(data.actuator_length

array([ 2.38005986e-02, -5.71048708e-02, -3.06039755e-01, -2.35761888e-02,
       -5.74202537e-02, -3.07561115e-01,  3.36956345e-02,  1.75751454e-02,
       -2.88085449e-01, -3.41002597e-02,  1.77964887e-02, -2.90205873e-01,
       -1.70292209e-05, -3.13826713e+00,  3.13420625e+00,  1.07701837e-04,
        2.74127678e-03, -1.01188188e-06,  1.84904872e-04])